## Monte Carlo Policy Control for Blackjack

In [4]:
import gymnasium as gym
import numpy as np
from collections import defaultdict

# Create the Blackjack environment
env = gym.make("Blackjack-v1", sab=True)


### Monte Carlo Policy Control Implementation

We will implement an on-policy Monte Carlo control algorithm, specifically first-visit Monte Carlo. This involves:
1.  **Initialization**: Start with an arbitrary policy and an empty dictionary to store Q-values (state-action values).
2.  **Episode Generation**: Generate an episode using the current policy.
3.  **Policy Evaluation**: For each state-action pair visited in the episode, update its Q-value based on the return (sum of future rewards).
4.  **Policy Improvement**: Update the policy to be greedy with respect to the new Q-values.

This process repeats for a specified number of episodes.

In [5]:
def monte_carlo_policy_control(env, num_episodes, gamma=1.0):
    """ Monte Carlo On-Policy Control (First-Visit) """

    # Initialize Q-values and returns
    Q = defaultdict(lambda: np.zeros(env.action_space.n))
    N = defaultdict(lambda: np.zeros(env.action_space.n)) # For tracking visits to state-action pairs

    # Initialize an arbitrary policy (e.g., stick if sum >= 20, else hit)
    # The policy will be improved, so the initial choice doesn't need to be optimal
    policy = defaultdict(lambda: np.random.choice(env.action_space.n))

    for i_episode in range(1, num_episodes + 1):
        if i_episode % 10000 == 0:
            print(f"Episode {i_episode}/{num_episodes}")

        episode = []
        state, info = env.reset()
        done = False

        while not done:
            player_sum, dealer_card, usable_ace = state

            # Choose action based on current policy (epsilon-greedy for exploration)
            # A simple way to initialize a policy for demonstration, it will be updated
            action = policy[state]

            # Take action
            next_state, reward, done, truncated, info = env.step(action)
            episode.append((state, action, reward))
            state = next_state

        # First-visit Monte Carlo policy evaluation
        G = 0 # Total return for the episode
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = gamma * G + reward

            # Check if (state, action) is visited for the first time in this episode
            if (state, action) not in [(x[0], x[1]) for x in episode[0:t]]:
                N[state][action] += 1
                Q[state][action] += (G - Q[state][action]) / N[state][action]

                # Policy Improvement (Greedy policy with respect to Q)
                # Update policy for this state to be greedy wrt Q
                policy[state] = np.argmax(Q[state])

    return policy, Q


In [3]:
# Run the Monte Carlo policy control
num_episodes = 500000
print(f"Running Monte Carlo Policy Control for {num_episodes} episodes...")
optimal_policy, optimal_q_values = monte_carlo_policy_control(env, num_episodes)
print("Monte Carlo Policy Control completed.")


Running Monte Carlo Policy Control for 500000 episodes...
Episode 10000/500000
Episode 20000/500000
Episode 30000/500000
Episode 40000/500000
Episode 50000/500000
Episode 60000/500000
Episode 70000/500000
Episode 80000/500000
Episode 90000/500000
Episode 100000/500000
Episode 110000/500000
Episode 120000/500000
Episode 130000/500000
Episode 140000/500000
Episode 150000/500000
Episode 160000/500000
Episode 170000/500000
Episode 180000/500000
Episode 190000/500000
Episode 200000/500000
Episode 210000/500000
Episode 220000/500000
Episode 230000/500000
Episode 240000/500000
Episode 250000/500000
Episode 260000/500000
Episode 270000/500000
Episode 280000/500000
Episode 290000/500000
Episode 300000/500000
Episode 310000/500000
Episode 320000/500000
Episode 330000/500000
Episode 340000/500000
Episode 350000/500000
Episode 360000/500000
Episode 370000/500000
Episode 380000/500000
Episode 390000/500000
Episode 400000/500000
Episode 410000/500000
Episode 420000/500000
Episode 430000/500000
Episo

### Displaying the Optimal Policy (for selected states)

The optimal policy indicates the best action (0 for 'stick', 1 for 'hit') for each state. Since there are many possible states, we'll display the policy for a few representative ones, specifically varying player sums, dealer's showing card, and usable ace.


In [7]:
# Helper function to visualize the policy
def print_policy(policy, usable_ace=False, player_range=(12, 21), dealer_range=(1, 10)):
    print(f"\nOptimal Policy (Usable Ace: {usable_ace})")
    print("Player Sum \\ Dealer Showing   ", end="")
    for dealer_card in range(dealer_range[0], dealer_range[1] + 1):
        print(f"{dealer_card:4}", end="")
    print()
    print("-" * (19 + 4 * (dealer_range[1] - dealer_range[0] + 1)))

    for player_sum in range(player_range[0], player_range[1] + 1):
        print(f"{player_sum:15}", end="")
        for dealer_card in range(dealer_range[0], dealer_range[1] + 1):
            state = (player_sum, dealer_card, usable_ace)
            action = policy[state] if state in policy else 0 # Default to stick if not visited
            print(f"{'H' if action == 1 else 'S':4}", end="") # H for hit, S for stick
        print()

# Print policies for different scenarios
print_policy(optimal_policy, usable_ace=False)
print_policy(optimal_policy, usable_ace=True)



Optimal Policy (Usable Ace: False)
Player Sum \ Dealer Showing      1   2   3   4   5   6   7   8   9  10
-----------------------------------------------------------
             12S   S   S   S   H   S   H   H   S   S   
             13S   S   H   S   S   S   S   H   S   S   
             14S   S   S   S   S   S   S   H   H   S   
             15S   S   S   S   S   S   H   H   S   S   
             16S   H   S   S   S   S   S   S   S   S   
             17H   S   S   S   H   S   S   S   S   S   
             18S   S   S   S   S   S   S   S   S   S   
             19S   S   S   S   S   S   S   S   S   S   
             20S   S   S   S   S   S   S   S   S   S   
             21S   S   S   S   S   S   S   S   S   S   

Optimal Policy (Usable Ace: True)
Player Sum \ Dealer Showing      1   2   3   4   5   6   7   8   9  10
-----------------------------------------------------------
             12S   S   S   S   H   H   S   H   S   S   
             13S   H   S   S   H   H   S   S   S   

### Interpreting the Results

*   **'H'** stands for **'Hit'**: Take another card.
*   **'S'** stands for **'Stick'**: Stand with the current hand.

This policy table shows the recommended action for various combinations of your hand total, the dealer's visible card, and whether you have a usable ace.

For example, if you have a total of 15 and the dealer is showing a 7 (and you don't have a usable ace), the policy will tell you whether to hit or stick based on the learned optimal strategy.